# 01 — Descoberta de estruturas no espaço de fase com Gaia

Workshop SAB — AI-Scope / ADSS

**Pergunta científica:** conseguimos encontrar uma população cinemática sem começar por uma lista de membros conhecida?

Vamos consultar uma região do céu no Gaia DR3 e explorar posição, paralaxe e movimento próprio. Em vez de selecionar uma população conhecida por cortes pré-definidos, o notebook usa agrupamento não supervisionado para procurar sobredensidades no espaço astrométrico.

> Para o workshop, ajuste `CENTER_RA`, `CENTER_DEC` e `RADIUS_DEG` para uma região de interesse com densidade adequada.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import adss

BASE_URL = "https://ai-scope.cbpf.br/"
USERNAME = os.getenv("AI_SCOPE_USERNAME")
PASSWORD = os.getenv("AI_SCOPE_PASSWORD")

if not USERNAME or not PASSWORD:
    raise RuntimeError("Defina AI_SCOPE_USERNAME e AI_SCOPE_PASSWORD no ambiente.")

cl = adss.ADSSClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD)

In [ ]:
CENTER_RA = 56.75
CENTER_DEC = 24.12
RADIUS_DEG = 1.0

query = f"""
SELECT TOP 50000
    source_id, ra, dec,
    parallax, parallax_error, parallax_over_error,
    pmra, pmdec, pmra_error, pmdec_error,
    phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag,
    bp_rp, ruwe
FROM public.gaia_dr3
WHERE cone(ra, dec, {CENTER_RA}, {CENTER_DEC}, {RADIUS_DEG})
  AND parallax IS NOT NULL
  AND pmra IS NOT NULL
  AND pmdec IS NOT NULL
  AND phot_g_mean_mag IS NOT NULL
"""

gaia = cl.query_and_wait(query_text=query, mode="astroql").data
gaia.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(gaia.pmra, gaia.pmdec, s=2, alpha=0.25)
ax.set(xlabel=r"$\mu_{\alpha*}$ [mas yr$^{-1}$]", ylabel=r"$\mu_\delta$ [mas yr$^{-1}$]", title="Proper-motion plane")
plt.show()

### Seleção de qualidade
Antes do agrupamento, removemos soluções astrométricas ruins. O corte abaixo é deliberadamente simples para que os participantes possam testar como a seleção altera o resultado.

In [ ]:
clean = gaia[(gaia.ruwe < 1.4) & (gaia.parallax_over_error > 5)].copy()
features = clean[["pmra", "pmdec", "parallax"]].dropna()
print(f"{len(features):,} fontes para clustering")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

X = StandardScaler().fit_transform(features)
labels = DBSCAN(eps=0.22, min_samples=20).fit_predict(X)
clean.loc[features.index, "cluster"] = labels
clean.cluster.value_counts().head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for label, group in clean[clean.cluster >= 0].groupby("cluster"):
    ax.scatter(group.pmra, group.pmdec, s=6, alpha=0.7, label=f"cluster {int(label)}")
ax.set(xlabel=r"$\mu_{\alpha*}$", ylabel=r"$\mu_\delta$", title="Astrometric overdensities")
if clean.loc[clean.cluster >= 0, "cluster"].nunique() <= 10:
    ax.legend(markerscale=2)
plt.show()

In [ ]:
# Diagrama cor-magnitude dos grupos encontrados
cand = clean[clean.cluster >= 0].copy()
cand["abs_g_proxy"] = cand.phot_g_mean_mag + 5*np.log10(cand.parallax/1000) + 5

fig, ax = plt.subplots(figsize=(6, 7))
ax.scatter(cand.bp_rp, cand.abs_g_proxy, c=cand.cluster, s=6, alpha=0.7)
ax.invert_yaxis()
ax.set(xlabel="BP - RP", ylabel="M_G (parallax proxy)", title="Do kinematic groups trace a stellar sequence?")
plt.show()

### Questões para discussão
Um agrupamento em movimento próprio é necessariamente um aglomerado? Como paralaxe, sequência no CMD e distribuição espacial ajudam a rejeitar falsos positivos? Que vieses são introduzidos por `parallax_over_error > 5`?